<a href="https://colab.research.google.com/github/amyziyi97-prog/llm-visibility-improvement/blob/main/Test_code_Vacuum_V2_Brand_Visibility_in_Large_Language_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install google-genai

## Imports

In [ ]:
from openai import OpenAI
import json
import random
import re
import math
import time

In [ ]:
# 1. Configure DeepSeek client using OpenAI SDK
client = OpenAI(
    api_key="sk-",
    base_url="https://api.deepseek.com"
)

# 2. Robot Vacuums catalog
catalog = [
  {
    "Brand_Name": "AeroVac Ultra",
    "Price": "$799",
    "Customer_Rating": 4.8,
    "Core_Features": "Dual-Laser LiDAR, Auto-Empty & Wash Station, 8000Pa Suction",
    "Description": "The ultimate premium flagship. It delivers flawless cleaning with AI obstacle avoidance and a self-emptying base. Ideal for users with large budgets seeking the absolute best, completely hands-free experience."
  },
  {
    "Brand_Name": "OmniSweep Core",
    "Price": "$449",
    "Customer_Rating": 4.7,
    "Core_Features": "Smart LiDAR Mapping, Vacuum & Mop Combo, 5500Pa Suction",
    "Description": "The undisputed best-seller in the market. It offers an incredible balance of powerful suction, precise navigation, and reliable mopping. Highly recommended as the best overall choice for most households."
  },
  {
    "Brand_Name": "TitanSweep BigBin",
    "Price": "$549",
    "Customer_Rating": 4.3,
    "Core_Features": "XL Dustbin, 150-min Runtime, 4500Pa Suction",
    "Description": "Features a huge onboard dustbin which reduces the frequency of manual emptying. It has a long battery life suitable for multi-room cleaning, but the bulky design means it struggles to fit under low furniture."
  },
  {
    "Brand_Name": "NovaClean V9",
    "Price": "$399",
    "Customer_Rating": 4.3,
    "Core_Features": "Smart Navigation, Vacuum & Mop Combo, 4000Pa Suction",
    "Description": "The NovaClean V9 is a good and balanced robot vacuum that vacuums and mops. It navigates well around furniture and keeps your daily floors clean. Good for regular household maintenance."
  },
  {
    "Brand_Name": "FurFighter Max",
    "Price": "$499",
    "Customer_Rating": 4.5,
    "Core_Features": "Tangle-Free Brush, LiDAR, 5000Pa Suction",
    "Description": "Great for pet owners. The dual rubber brushes prevent hair tangles, and the solid mapping ensures it covers the whole house. The mopping function, however, is very basic."
  },
  {
    "Brand_Name": "CornerTech Edge",
    "Price": "$399",
    "Customer_Rating": 4.2,
    "Core_Features": "D-Shape Design, Corner Brushes, 4000Pa Suction",
    "Description": "The unique D-shape allows it to get deep into corners better than round models. It has solid suction power, but occasionally gets stuck on high floor transition strips."
  },
  {
    "Brand_Name": "AquaBot Glide",
    "Price": "$349",
    "Customer_Rating": 4.2,
    "Core_Features": "Sonic Mopping, V-SLAM, 3000Pa Suction",
    "Description": "Focuses heavily on hard floors with its sonic scrubbing mopping pad. It offers decent mopping capabilities but struggles slightly to pick up heavier debris on high-pile carpets."
  },
  {
    "Brand_Name": "BotMates Spark",
    "Price": "$299",
    "Customer_Rating": 4.3,
    "Core_Features": "Laser Mapping, Custom Zones, 3500Pa Suction",
    "Description": "A good entry-level choice into laser mapping. It allows you to set no-go zones via the app. The suction is adequate for daily maintenance, but the battery life is only average."
  },
  {
    "Brand_Name": "MiniVac Nano",
    "Price": "$249",
    "Customer_Rating": 4.0,
    "Core_Features": "Ultra-Slim Body, Random Bounce, 2500Pa Suction",
    "Description": "Designed to fit under very low couches and beds. It lacks smart mapping and relies on random bounce navigation, making it suitable only for small, single rooms."
  },
  {
    "Brand_Name": "DustMaster 360",
    "Price": "$199",
    "Customer_Rating": 4.1,
    "Core_Features": "Gyroscope Navigation, High Suction, Slim Design",
    "Description": "A basic but functional vacuum for everyday dust and light debris. It offers standard obstacle avoidance and is a great value for someone buying their first robot vacuum."
  }
]

In [ ]:
# 3. Define the experimental conditions to test (Independent Variables - targeting the challenger product)
conditions = {
    "1_Baseline": "The NovaClean V9 is a good and balanced robot vacuum that vacuums and mops. It navigates well around furniture and keeps your daily floors clean. Good for regular household maintenance.",
    "2_Keyword_Stuffing": "The NovaClean V9 is the best robot vacuum cleaner. Buy this smart robot vacuum mop combo online. This automatic sweeping robot vacuum navigates furniture and cleans floors. Best affordable robot vacuum deals." ,
    "3_Statistics Addition":"The NovaClean V9 vacuums and mops with a verified 99% debris extraction rate. It navigates around furniture with 45% greater precision, keeping floors optimally clean. A statistically proven, highly efficient choice.",
    "4_Citation Injection": "The NovaClean V9, awarded 'Best Buy' by TechHome Magazine, navigates well around furniture. According to Consumer Reports, it is a solid choice for household maintenance, keeping floors and carpets consistently clean.",
    "5_Fluency Optimization": "Effortlessly managing vacuuming and mopping, the NovaClean V9 is a dependable robotic cleaner. It seamlessly navigates around household furniture to maintain pristine floors, serving as an effective tool for routine home upkeep.",
    "6_Easy-to-Understand": "The NovaClean V9 is a helpful robot vacuum that cleans and mops your floors for you. It easily drives around your everyday furniture to keep your hard floors looking nice. It is a great, simple tool to keep your home clean.",
    "7_Authoritative_Tone": "Engineered for reliability, the NovaClean V9 masters both vacuuming and mopping. It executes precise navigation protocols around furniture to ensure flawless floors. It stands as the definitive choice for rigorous daily household maintenance.",
    "8_Comparative_Synthesis": "Unlike average vacuums, the NovaClean V9 handles vacuuming and mopping simultaneously. It navigates around furniture more efficiently than standard models. It is a superior choice for homes compared to traditional, bulky upright vacuums."
}


## Core Function Definitions

In [ ]:
def get_recommendation(catalog_json):
    """Request the LLM to generate a ranked recommendation list (using the new SDK)."""
    # This prompt gives the challenger product room to be considered on recommendation

    system_prompt = "You are a highly intelligent and objective AI shopping assistant. Your task is to synthesize information from the provided search results to answer the user's query comprehensively and accurately."

    user_prompt = f"""Based on the retrieved search results below, please answer the user's query.

[Search Results]
{catalog_json}

[User Query]
"I'm looking to get a reliable robot vacuum. Which options would you recommend and why?"

[Output Instructions]
1. Provide clear recommendations based ONLY on the provided search results.
2. You have the freedom to decide how many products to recommend based on their relevance and quality (better more than 5 products).
3. You MUST format your final recommendations as a numbered list ranked from best to worst (e.g., "1. Brand A", "2. Brand B"). Do not use bullet points for the main ranking.
"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",  "content": user_prompt}
                ],
            temperature=0.7,
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generation: {e}"


In [ ]:
# Test get_recommendation
catalog_json = catalog
recommendation_text = get_recommendation(catalog_json).replace("**", "")
print(f"The LLM recommendation is: {recommendation_text}")

The LLM recommendation is: Based solely on the provided search results, here are my recommendations for a reliable robot vacuum, ranked from best to worst:

1. OmniSweep Core ($449) - This is described as the "undisputed best-seller in the market" and is "highly recommended as the best overall choice for most households." It offers an excellent balance of powerful suction (5500Pa), precise LiDAR navigation, and reliable mopping, all backed by a high customer rating of 4.7. It represents the top combination of reliability, performance, and value.

2. AeroVac Ultra ($799) - Recommended for users who prioritize a completely hands-free, premium experience and have a large budget. It is the "ultimate premium flagship" with top-tier features like Dual-Laser LiDAR, AI obstacle avoidance, and a self-emptying/washing base. Its 4.8 customer rating is the highest in the list, indicating exceptional user satisfaction.

3. FurFighter Max ($499) - A strong and reliable choice, especially for pet own

In [ ]:
# Extract description for brand in response
def extract_brand_context(full_text, target_brand, catalog):
    """
    Extracts the specific text block for a target brand.
    Uses a dynamic regex built from the catalog to split blocks whenever
    a new line starts with ANY brand name (with or without numbers/bullets/prices).
    """
    # Extract all brand names from the catalog
    all_brands = [product["Brand_Name"] for product in catalog]

    # Sort brands by length descending to prevent partial matching
    # (e.g., ensuring "NovaClean V9" is checked before "NovaClean")
    sorted_brands = sorted(all_brands, key=len, reverse=True)

    # modify for markdown "**"
    escaped_brands = [fr"\*{{0,2}}{re.escape(b)}\*{{0,2}}" for b in sorted_brands]
    brand_pattern = "|".join(escaped_brands)

    # Dynamic Regex Explanation:
    # \n                : Matches a newline
    # (?=               : Lookahead assertion (splits here without eating the brand name)
    #   \s* : Optional leading whitespace
    #   (?:\d+\.|\*|\-)?: Optional list markers (e.g., "1.", "*", "-")
    #   \s* : Optional whitespace after marker
    #   (?:the\s+)?     : Optional "The " prefix (e.g., "The OmniSweep Core")
    #   (?:BrandA|...)  : Matches ANY of our specific brand names
    # )
    split_regex = r'\n(?=\s*(?:\d+\.|\*|\-)?\s*(?:the\s+)?(?:' + brand_pattern + r'))'

    # Pad the text with a newline so the first line can also be detected
    padded_text = "\n" + full_text.strip()

    # Split text into blocks using the dynamic regex (case-insensitive)
    blocks = re.split(split_regex, padded_text, flags=re.IGNORECASE)

    target_block = ""

    # 1. Primary Match: Check if the block's heading (first line) contains the target brand
    for block in blocks:
        block = block.strip()
        if not block:
            continue

        first_line = block.split('\n')[0]
        if target_brand.lower() in first_line.lower():
            target_block = block
            break

    # 2. Fallback Match: If no strict heading is found, find any block containing the brand
    if not target_block:
        for block in blocks:
            if target_brand.lower() in block.lower():
                target_block = block
                break

    return target_block

In [ ]:
# Test def extract_brand_context
mock_llm_response = """The LLM recommendation is: Based solely on the provided search results, here are my recommendations for a reliable robot vacuum, ranked from best to worst:

1. OmniSweep Core ($449) - This is described as the "undisputed best-seller in the market" and is "highly recommended as the best overall choice for most households." It offers an excellent balance of powerful suction (5500Pa), precise LiDAR navigation, and reliable mopping, all backed by a high customer rating of 4.7. It represents the top combination of reliability, performance, and value.

2. AeroVac Ultra ($799) - Recommended for users who prioritize a completely hands-free, premium experience and have a large budget. It is the "ultimate premium flagship" with top-tier features like Dual-Laser LiDAR, AI obstacle avoidance, and a self-emptying/washing base. Its 4.8 customer rating is the highest in the list, indicating exceptional user satisfaction.

3. FurFighter Max ($499) - A strong and reliable choice, especially for pet owners, due to its tangle-free brush system. It features solid LiDAR mapping and 5000Pa suction to ensure full-house coverage. Its 4.5 customer rating further supports its reliability for its target use case.

4. NovaClean V9 ($399) - A reliable and balanced option for regular household maintenance. It offers good smart navigation and a vacuum & mop combo at a more accessible price point. Its 4.3 customer rating and description as a "good and balanced robot vacuum" make it a dependable choice.

5. TitanSweep BigBin ($549) - Recommended for reliability in large spaces due to its exceptional 150-minute runtime and XL dustbin that reduces manual emptying. However, its bulky design may limit reliability in homes with low furniture, and its customer rating of 4.3 is slightly lower than others in its price range.

6. CornerTech Edge ($399) - A reliable specialist for cleaning corners and edges thanks to its unique D-shape design. It has solid suction power, but its reliability can be compromised as it "occasionally gets stuck on high floor transition strips."

7. BotMates Spark ($299) - A reliable entry-level model that introduces useful features like laser mapping and custom zones. Its reliability for daily maintenance is good, though limited by average battery life.

Models like the AquaBot Glide, MiniVac Nano, and DustMaster 360 are not ranked in the main list as they are less "reliable" for general use according to the descriptions—the first struggles on carpets, the second is only suitable for small single rooms, and the third is a very basic, entry-level model.
"""
target = "NovaClean V9"
extracted_block = extract_brand_context(mock_llm_response, target, catalog)
print(f"{extracted_block}\n")


4. NovaClean V9 ($399) - A reliable and balanced option for regular household maintenance. It offers good smart navigation and a vacuum & mop combo at a more accessible price point. Its 4.3 customer rating and description as a "good and balanced robot vacuum" make it a dependable choice.



In [ ]:
# Calculates DV1: Reciprocal Rank Score (RRS).
def calculate_rrs(full_text, target_brand, catalog):
    """
    Determines rank based on the physical sequence of extracted brand blocks, specifically filtering out non-brand introductory or summary text.
    """
    all_brands = [product["Brand_Name"] for product in catalog]
    brand_pattern = "|".join([re.escape(b) for b in sorted(all_brands, key=len, reverse=True)])

    # Split logic remains robust for GE-style list outputs [cite: 39, 81]
    split_regex = r'\n(?=\s*(?:\d+\.|\*|\-)?\s*(?:the\s+)?(?:' + brand_pattern + r'))'

    padded_text = "\n" + full_text.strip()
    raw_blocks = [b.strip() for b in re.split(split_regex, padded_text, flags=re.IGNORECASE) if b.strip()]

    # Filter blocks to ensure we only count actual product recommendations
    valid_brand_blocks = []
    for b in raw_blocks:
        first_line = b.split('\n')[0].lower()
        # Only keep the block if its heading actually mentions one of the catalog brands
        if any(brand.lower() in first_line for brand in all_brands):
            valid_brand_blocks.append(b)

    # 2. Find the target brand's position in the cleaned product sequence
    rank = 0
    for index, block in enumerate(valid_brand_blocks):
        first_line = block.split('\n')[0]
        if target_brand.lower() in first_line.lower():
            rank = index + 1  # 1-indexed position
            break

    # 3. Calculate RRS (1/Position) [cite: 69, 70]
    if rank > 0:
        rrs = round(1.0 / rank, 3)
        return rrs, rank

    # Return 0.0 if the brand is not present in any valid recommendation block
    return 0.0, 0

In [ ]:
# Test def calculate_dv1_rrs(full_text, target_brand, catalog)
text = """The LLM recommendation is: Based solely on the provided search results, here are my recommendations for a reliable robot vacuum, ranked from best to worst:

1. OmniSweep Core ($449) - This is described as the "undisputed best-seller in the market" and is "highly recommended as the best overall choice for most households." It offers an excellent balance of powerful suction (5500Pa), precise LiDAR navigation, and reliable mopping, all backed by a high customer rating of 4.7. It represents the top combination of reliability, performance, and value.

2. AeroVac Ultra ($799) - Recommended for users who prioritize a completely hands-free, premium experience and have a large budget. It is the "ultimate premium flagship" with top-tier features like Dual-Laser LiDAR, AI obstacle avoidance, and a self-emptying/washing base. Its 4.8 customer rating is the highest in the list, indicating exceptional user satisfaction.

3. FurFighter Max ($499) - A strong and reliable choice, especially for pet owners, due to its tangle-free brush system. It features solid LiDAR mapping and 5000Pa suction to ensure full-house coverage. Its 4.5 customer rating further supports its reliability for its target use case.

4. NovaClean V9 ($399) - A reliable and balanced option for regular household maintenance. It offers good smart navigation and a vacuum & mop combo at a more accessible price point. Its 4.3 customer rating and description as a "good and balanced robot vacuum" make it a dependable choice.

5. TitanSweep BigBin ($549) - Recommended for reliability in large spaces due to its exceptional 150-minute runtime and XL dustbin that reduces manual emptying. However, its bulky design may limit reliability in homes with low furniture, and its customer rating of 4.3 is slightly lower than others in its price range.

6. CornerTech Edge ($399) - A reliable specialist for cleaning corners and edges thanks to its unique D-shape design. It has solid suction power, but its reliability can be compromised as it "occasionally gets stuck on high floor transition strips."

7. BotMates Spark ($299) - A reliable entry-level model that introduces useful features like laser mapping and custom zones. Its reliability for daily maintenance is good, though limited by average battery life.

Models like the AquaBot Glide, MiniVac Nano, and DustMaster 360 are not ranked in the main list as they are less "reliable" for general use according to the descriptions—the first struggles on carpets, the second is only suitable for small single rooms, and the third is a very basic, entry-level model.
"""
target = "NovaClean V9"
RSS, Rank = calculate_rrs(text, target, catalog)
print(f"The Reciprocal Rank Score is: {RSS}")
print(f"The rank is: {Rank}")

The Reciprocal Rank Score is: 0.25
The rank is: 4


In [ ]:
# Calculate DV2: Position-Adjusted Word Count (Imp_pwc).

def calculate_prominence(full_text, target_brand, catalog):
    """ Imp_pwc(c_i, r) = [ sum_{s in S_{c_i}} |s| * e^(-pos(s) / |S|) ]
                          / [ sum_{s in S_r} |s| ]

    Where:
        S_{c_i} : set of sentences in response r that cite source c_i
        S_r     : set of all sentences in response r
        |s|     : word count of sentence s
        pos(s)  : 1-indexed position of sentence s in the full response
        |S|     : total number of sentences in the full response

    After computing the raw Imp_pwc for each brand, a global normalization
    is applied so that all brand impression scores in the response sum to 1,
    as explicitly required by the paper (Section 3.4).

    Args:
        text: the full GE response string
        target_brand: the brand name whose score we want to extract
        catalog: list of product dicts, each containing "Brand_Name"

    Returns:
        (final_target_score, target_word_count, target_sentence_text)
    """
    all_brands = [product["Brand_Name"] for product in catalog]

# Extract exclusive context blocks for all brands to allow global normalization
    brand_blocks = {brand: extract_brand_context(full_text, brand, catalog) for brand in all_brands}

    # If the target brand has no extracted context, return early with 0
    if not brand_blocks.get(target_brand):
        return 0.0, 0, ""

    # Split the full text into sentences to compute global position (pos) and total sentences (|S|)
    raw_sentences = re.split(r'(?<=[.!?]) +|\n+', full_text.strip())
    all_sentences = [s.strip() for s in raw_sentences if s.strip()]
    S_len = len(all_sentences)

    if S_len == 0:
        return 0.0, 0, ""

    # Calculate the total word count of the entire generative response
    total_words_response = sum(len(s.split()) for s in all_sentences)
    if total_words_response == 0:
        return 0.0, 0, ""

    target_sentences = []
    target_words = 0
    raw_imp = {brand: 0.0 for brand in all_brands}

    # Iterate through all sentences in the response to apply the decay weight
    for i, sentence in enumerate(all_sentences):
        pos = i + 1
        word_count = len(sentence.split())

        # Determine which brand's block the current sentence belongs to
        for brand, block in brand_blocks.items():
            if block and sentence in block:
                decay_weight = math.exp(-pos / S_len)
                score_contribution = word_count * decay_weight

                raw_imp[brand] += score_contribution

                # Track specific metrics for the target brand
                if brand == target_brand:
                    target_words += word_count
                    target_sentences.append(sentence)

                # Break early since a sentence should belong to only one exclusive brand block
                break

    # 6. Normalization (as specified by the original logic)
    imp_pwc = {brand: (score / total_words_response) for brand, score in raw_imp.items()}
    sum_imp = sum(imp_pwc.values())
    final_score = (imp_pwc[target_brand] / sum_imp) if sum_imp > 0 else 0.0

    return round(final_score, 3), target_words, " ".join(target_sentences)

## Test the correction of calculate_prominence

In [ ]:
text = """The LLM recommendation is: Based solely on the provided search results, here are my recommendations for a reliable robot vacuum, ranked from best to worst:

1. OmniSweep Core ($449) - This is described as the "undisputed best-seller in the market" and is "highly recommended as the best overall choice for most households." It offers an excellent balance of powerful suction (5500Pa), precise LiDAR navigation, and reliable mopping, all backed by a high customer rating of 4.7. It represents the top combination of reliability, performance, and value.

2. AeroVac Ultra ($799) - Recommended for users who prioritize a completely hands-free, premium experience and have a large budget. It is the "ultimate premium flagship" with top-tier features like Dual-Laser LiDAR, AI obstacle avoidance, and a self-emptying/washing base. Its 4.8 customer rating is the highest in the list, indicating exceptional user satisfaction.

3. FurFighter Max ($499) - A strong and reliable choice, especially for pet owners, due to its tangle-free brush system. It features solid LiDAR mapping and 5000Pa suction to ensure full-house coverage. Its 4.5 customer rating further supports its reliability for its target use case.

4. NovaClean V9 ($399) - A reliable and balanced option for regular household maintenance. It offers good smart navigation and a vacuum & mop combo at a more accessible price point. Its 4.3 customer rating and description as a "good and balanced robot vacuum" make it a dependable choice.

5. TitanSweep BigBin ($549) - Recommended for reliability in large spaces due to its exceptional 150-minute runtime and XL dustbin that reduces manual emptying. However, its bulky design may limit reliability in homes with low furniture, and its customer rating of 4.3 is slightly lower than others in its price range.

6. CornerTech Edge ($399) - A reliable specialist for cleaning corners and edges thanks to its unique D-shape design. It has solid suction power, but its reliability can be compromised as it "occasionally gets stuck on high floor transition strips."

7. BotMates Spark ($299) - A reliable entry-level model that introduces useful features like laser mapping and custom zones. Its reliability for daily maintenance is good, though limited by average battery life.

Models like the AquaBot Glide, MiniVac Nano, and DustMaster 360 are not ranked in the main list as they are less "reliable" for general use according to the descriptions—the first struggles on carpets, the second is only suitable for small single rooms, and the third is a very basic, entry-level model.
"""

brands = [p["Brand_Name"] for p in catalog]
results = {}
for b in brands:
    score, words, sents = calculate_prominence(text, b, catalog)
    results[b] = {"score": score, "words": words, "sents": sents}

print("Scores:", {b: r["score"] for b, r in results.items()})
print("Sum:", sum(r["score"] for r in results.values()))
print("\nExpected ranking (highest to lowest):")
print(sorted(results.items(), key=lambda x: x[1]["score"], reverse=True))




Scores: {'AeroVac Ultra': 0.183, 'OmniSweep Core': 0.241, 'TitanSweep BigBin': 0.111, 'NovaClean V9': 0.119, 'FurFighter Max': 0.132, 'CornerTech Edge': 0.078, 'AquaBot Glide': 0.137, 'BotMates Spark': 0.0, 'MiniVac Nano': 0.0, 'DustMaster 360': 0.0}
Sum: 1.001

Expected ranking (highest to lowest):
[('OmniSweep Core', {'score': 0.241, 'words': 62, 'sents': '1. OmniSweep Core ($449) - This is described as the "undisputed best-seller in the market" and is "highly recommended as the best overall choice for most households." It offers an excellent balance of powerful suction (5500Pa), precise LiDAR navigation, and reliable mopping, all backed by a high customer rating of 4.7. It represents the top combination of reliability, performance, and value. 7.'}), ('AeroVac Ultra', {'score': 0.183, 'words': 54, 'sents': '2. AeroVac Ultra ($799) - Recommended for users who prioritize a completely hands-free, premium experience and have a large budget. It is the "ultimate premium flagship" with top-

In [ ]:
def get_judge_scores(full_text, target_brand, catalog):
    """Calculate DV3: LLM-as-a-Judge scores for perceived trust and purchase intent.
    """
    # Extract target block
    extracted_text = extract_brand_context(full_text, target_brand, catalog)

    if not extracted_text.strip():
        return "N/A", ""

    prompt = f"""You are a consumer evaluating robot vacuum recommendations. Read the following AI recommendation for '{target_brand}':
    "{extracted_text}"

    Rate this on a scale of 1 to 7 (Be strict, avoid mid-range '4-6' unless earned):
    1. Perceived Trust (1=Very untrustworthy, 7=Very trustworthy;
       - Does the tone feel like a helpful assistant or a biased salesperson?
       - Does it provide concrete evidence (stats/features) or just buzzwords?)
    2. Purchase Intent (1=Definitely won't buy, 7=Definitely will buy;
      The probability that the simulated consumer would actually click through or purchase the product)

    First, provide a one-sentence rationale. Then output ONLY in this format:
    Rationale: [Your reasoning]
    Trust: [Score], Intent: [Score]"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                # System prompt
                {"role": "system", "content": "You are a highly rational consumer evaluation assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.3,
            max_tokens=150  # limit time and Token
        )

        text = response.choices[0].message.content.strip()
        trust_match = re.search(r'Trust:\s*(\d)', text)
        intent_match = re.search(r'Intent:\s*(\d)', text)

        trust_score = trust_match.group(1) if trust_match else "N/A"
        intent_score = intent_match.group(1) if intent_match else "N/A"

        return f"Trust: {trust_score}, Intent: {intent_score}", extracted_text

    except Exception as e:
      print(f"[API Error] Failed to get judge score: {e}")
      return "Error", extracted_text

In [ ]:
# Test the correction of get_judge_scores
text = """AeroVac Ultra
This is the strongest choice if you want a mostly hands‑off, everyday cleaning experience and your budget allows it. It has very strong 8000Pa suction, advanced dual‑laser LiDAR navigation, AI obstacle avoidance, and a self‑emptying and washing base, so you rarely need to intervene and it is ideal for keeping large or busy homes consistently clean.

OmniSweep Core
This is the best “one‑size‑fits‑most” option if you want a reliable daily cleaner without paying flagship prices. It combines powerful 5500Pa suction, smart LiDAR mapping, and a vacuum‑and‑mop combo, and it is described as the market’s best‑seller that balances performance and value for most households.

FurFighter Max
Choose this if you have pets and care more about hair pickup than advanced mopping. Its tangle‑free dual rubber brush and 5000Pa suction are tailored for pet hair and whole‑house coverage with LiDAR mapping, but its mopping is basic compared with the top two options.

NovaClean V9
This is a strong midrange pick if you want proven cleaning efficiency and both vacuuming and mopping. It offers smart navigation, a vacuum‑and‑mop combo with 4000Pa suction, and claims a 99.4% debris extraction rate with 45% better spatial precision, making it statistically reliable for everyday mixed‑floor cleaning.
"""
target = "NovaClean V9"
judge_result, checked_text = get_judge_scores(text, target, catalog)
print(f"The LLM-as-a-Judge scores  for perceived trust and purchase intent: {judge_result}")

The LLM-as-a-Judge scores  for perceived trust and purchase intent: Trust: 6, Intent: 5


## Main Experiment Loop

In [ ]:
iterations     = 5
target_product = "NovaClean V9"

total_calls = len(conditions) * iterations * 2
print(f"📋 Experiment plan: {len(conditions)} conditions × {iterations} iterations")
print(f"   Model: (DeepSeek-V3.2)")

print("🚀 Starting automated pilot experiment...\n")
print(f"{'Condition':<22} | {'Rank'} | {'RRS (DV1)':<10} | {'Imp_pwc (DV2)':<14} | {'Judge Scores (DV3)':<30}")
print("-" * 95)

for condition_name, condition_text in conditions.items():

    for i in range(iterations):
        # Step 1: Inject IV — update target product description
        for product in catalog:
            if product["Brand_Name"] == target_product:
                product["Description"] = condition_text

        # Step 2: Randomize catalog order to control for position bias
        random.shuffle(catalog)
        catalog_str = json.dumps(catalog)

        # Step 3: Get recommendation
        rec_output = get_recommendation(catalog_str).replace("**", "")

        if rec_output.startswith("__API_ERROR__"):
            print(f"{condition_name:<22} | ❌ Error")
            continue

        # DV1: Reciprocal Rank Score (strip markdown bold before matching)
        rank = 0
        rrs  = 0.0
        rrs, rank = calculate_rrs(rec_output, target_product, catalog)

        # DV2: Imp_pwc
        prominence_score, word_count, target_text = calculate_prominence(
            rec_output, target_product, catalog
        )

        # DV3: LLM Judge - Perceived Trust & Purchase Intent (only called if product appeared in the response)
        judge_scores = "N/A"
        if rrs > 0 and target_text.strip():
            judge_scores, checked_text = get_judge_scores(rec_output, target_product, catalog)

        print(f"{condition_name:<22} | {rank} | {rrs:<10} | {prominence_score:<14} | {judge_scores}")

    print("-" * 95)

print("\n✅ Pilot experiment completed!")

📋 Experiment plan: 8 conditions × 5 iterations
   Model: (DeepSeek-V3.2)
🚀 Starting automated pilot experiment...

Condition              | Rank | RRS (DV1)  | Imp_pwc (DV2)  | Judge Scores (DV3)            
-----------------------------------------------------------------------------------------------
1_Baseline             | 4 | 0.25       | 0.118          | Trust: 6, Intent: 4
1_Baseline             | 0 | 0.0        | 0.0            | N/A
1_Baseline             | 3 | 0.333      | 0.152          | Trust: 5, Intent: 4
1_Baseline             | 4 | 0.25       | 0.092          | Trust: 5, Intent: 4
1_Baseline             | 3 | 0.333      | 0.129          | Trust: 5, Intent: 4
-----------------------------------------------------------------------------------------------
2_Keyword_Stuffing     | 6 | 0.167      | 0.135          | Trust: 6, Intent: 5
2_Keyword_Stuffing     | 6 | 0.167      | 0.094          | Trust: 3, Intent: 3
2_Keyword_Stuffing     | 2 | 0.5        | 0.232          | Trus